# Test Data Processing, Two-Tower Inference & Recommendations

### Pipeline Overview:
1. **Load Test Data**: Reads `test_source1.tsv`, `test_source2.tsv`, and `test_source3.tsv` from `dataset/test/`.
2. **Preprocessing**: Normalizes raw text into model-ready features (`name_norm`, `name_ascii`, `address_norm`, `country_norm`, `legal_form`).
3. **Load Model Weights**: Restores trained Character-CNN Two-Tower encoder (`best.pt`, `tokenizer.json`, `legal_form_vocab.json`).
4. **Candidate Pool Embeddings**: Encodes all Source 2 & Source 3 test records into 128-D normalized vectors.
5. **Interactive Recommendations**: Query any test entity ID to view Top-K candidate matches and similarity scores.
6. **Submission Generation**: Generates `matching_results.tsv` matching the competition submission format.


In [ ]:
import sys
import os
import time
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# Compute device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# Project root resolution
notebook_dir = Path.cwd().resolve()
for parent in [notebook_dir, *notebook_dir.parents]:
    if (parent / "dataset" / "test").exists():
        PROJECT_ROOT = parent
        break
else:
    PROJECT_ROOT = notebook_dir

# Test data paths
TEST_DIR = PROJECT_ROOT / "dataset" / "test"
TEST_S1_PATH = TEST_DIR / "test_source1.tsv"
TEST_S2_PATH = TEST_DIR / "test_source2.tsv"
TEST_S3_PATH = TEST_DIR / "test_source3.tsv"

# Trained model weights directory (checks src/checkpoints, models/two_tower_model, or kaggle)
possible_weight_dirs = [
    PROJECT_ROOT / "src" / "checkpoints",
    PROJECT_ROOT / "models" / "two_tower_model",
    Path("C:/kaggle/working/two_tower_model"),
    Path("/kaggle/working/two_tower_model"),
]
MODEL_DIR = next((d for d in possible_weight_dirs if (d / "best.pt").exists()), possible_weight_dirs[0])

# Output directory for results
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Number of rows to test (Set to None for full test dataset)
SAMPLE_ROWS = 5000

print(f"[INFO] Test S1 Path : {TEST_S1_PATH}")
print(f"[INFO] Test S2 Path : {TEST_S2_PATH}")
print(f"[INFO] Test S3 Path : {TEST_S3_PATH}")
print(f"[INFO] Model Weights: {MODEL_DIR}")
print(f"[INFO] Output Dir   : {OUTPUT_DIR}")


In [ ]:
TEXT_FIELDS = ("name_norm", "name_ascii", "address_norm", "country_norm")

def split_legal_forms(value: str) -> list[str]:
    value = str(value or "").strip()
    if value.startswith("["):
        try:
            value = ",".join(json.loads(value))
        except (json.JSONDecodeError, TypeError):
            pass
    strip_chars = " '\""
    tokens = re.split(r"[,;|]", value)
    return [x.strip(strip_chars) for x in tokens if x.strip(strip_chars) and x.strip(strip_chars) != "missing"]


class CharTokenizer:
    PAD, UNK = 0, 1

    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab

    def encode(self, values, length: int) -> torch.Tensor:
        output = np.zeros((len(values), length), dtype=np.int64)
        for row, value in enumerate(values):
            ids = [self.vocab.get(char, self.UNK) for char in str(value)[:length]]
            output[row, :len(ids)] = ids
        return torch.from_numpy(output)

    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))


class LegalFormVocab:
    PAD, UNK = 0, 1

    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab

    def encode(self, values) -> torch.Tensor:
        forms = [split_legal_forms(value) for value in values]
        width = max(1, max(map(len, forms), default=0))
        output = np.zeros((len(forms), width), dtype=np.int64)
        for row, values_for_row in enumerate(forms):
            if values_for_row:
                output[row, :len(values_for_row)] = [self.vocab.get(value, self.UNK) for value in values_for_row]
        return torch.from_numpy(output)

    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))


class CharCNN(nn.Module):
    def __init__(self, vocab_size: int, char_dim: int = 48, channels: int = 64, output_dim: int = 128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, char_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(char_dim, channels, kernel_size=kernel, padding=kernel // 2)
            for kernel in (3, 5, 7)
        ])
        self.projection = nn.Linear(channels * 3, output_dim)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        present = token_ids.ne(0)
        x = self.embedding(token_ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            value = F.relu(conv(x)).masked_fill(~present.unsqueeze(1), -1e4)
            value = value.max(dim=-1).values
            pooled.append(torch.where(present.any(1, keepdim=True), value, torch.zeros_like(value)))
        return self.projection(torch.cat(pooled, dim=1))


class EntityEncoder(nn.Module):
    def __init__(self, char_vocab_size: int, legal_vocab_size: int,
                 char_dim: int = 48, channels: int = 64, field_dim: int = 128,
                 legal_dim: int = 16, output_dim: int = 128, dropout: float = 0.15):
        super().__init__()
        self.name_cnn = CharCNN(char_vocab_size, char_dim, channels, field_dim)
        self.address_cnn = CharCNN(char_vocab_size, char_dim, channels, field_dim)
        self.legal_embedding = nn.Embedding(legal_vocab_size, legal_dim, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(field_dim * 4 + legal_dim, 256),
            nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, output_dim),
        )

    def forward(self, batch: dict[str, torch.Tensor]) -> torch.Tensor:
        legal_ids = batch["legal_form"]
        legal_vectors = self.legal_embedding(legal_ids)
        legal_mask = legal_ids.ne(0).unsqueeze(-1)
        legal = (legal_vectors * legal_mask).sum(1) / legal_mask.sum(1).clamp_min(1)
        features = torch.cat([
            self.name_cnn(batch["name_norm"]),
            self.name_cnn(batch["name_ascii"]),
            self.address_cnn(batch["address_norm"]),
            self.address_cnn(batch["country_norm"]),
            legal,
        ], dim=1)
        return F.normalize(self.mlp(features), p=2, dim=1)


def records_to_batch(records: pd.DataFrame, tokenizer: CharTokenizer,
                     legal_vocab: LegalFormVocab, device, name_length: int = 112,
                     address_length: int = 224, country_length: int = 32):
    return {
        "name_norm": tokenizer.encode(records["name_norm"], name_length).to(device),
        "name_ascii": tokenizer.encode(records["name_ascii"], name_length).to(device),
        "address_norm": tokenizer.encode(records["address_norm"], address_length).to(device),
        "country_norm": tokenizer.encode(records["country_norm"], country_length).to(device),
        "legal_form": legal_vocab.encode(records["legal_form"]).to(device),
    }


@torch.inference_mode()
def generate_embeddings(model, tokenizer, legal_vocab, records: pd.DataFrame, device, batch_size: int = 512):
    output = []
    model.eval()
    for start in range(0, len(records), batch_size):
        batch = records_to_batch(records.iloc[start:start + batch_size], tokenizer, legal_vocab, device)
        output.append(model(batch).cpu())
    return torch.cat(output).numpy() if output else np.empty((0, 128), dtype=np.float32)

print("[INFO] Model architecture ready.")


In [ ]:
# Legal Form Normalization Dictionary
LEGAL_FORMS_MAP = {
    "private limited": "private_limited", "pvt limited": "private_limited",
    "private ltd": "private_limited", "pvt ltd": "private_limited",
    "pvtltd": "private_limited", "prvt ltd": "private_limited",
    "limited liability company": "llc", "llc": "llc", "l l c": "llc",
    "limited liability partnership": "llp", "llp": "llp", "l l p": "llp",
    "corporation": "corp", "corp": "corp", "incorporated": "inc", "inc": "inc",
    "limited": "ltd", "ltd": "ltd", "co": "company", "company": "company",
    "societe anonyme": "sa", "sa": "sa", "sarl": "sarl", "sas": "sas",
    "gmbh": "gmbh", "ag": "ag", "trust": "trust", "foundation": "foundation"
}

_SORTED_PATTERNS = sorted(LEGAL_FORMS_MAP.keys(), key=len, reverse=True)
_LEGAL_REGEX = re.compile(
    r'(?:(?<=\s)|(?<=^)|(?<=\b))(' + '|'.join(re.escape(k) for k in _SORTED_PATTERNS) + r')(?:(?=\s)|(?=$)|(?=\b))',
    re.IGNORECASE
)
_APOSTROPHE_REGEX = re.compile(r"['’`]")
_PUNCT_REGEX = re.compile(r'[^\w\s\u0900-\u0D7F]', re.UNICODE)
_WHITESPACE_REGEX = re.compile(r'\s+')
_ASCII_CLEAN_REGEX = re.compile(r'[^a-z0-9\s]')

def preprocess_test_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Transforms raw test columns (business_name, business_address, country) into model features."""
    names = df["business_name"].fillna("").astype(str).tolist()
    addresses = df["business_address"].fillna("").astype(str).tolist()
    countries = df["country"].fillna("").astype(str).tolist()

    cleaned_name_norm = []
    cleaned_name_ascii = []
    cleaned_legal_form = []
    cleaned_address_norm = []
    cleaned_country_norm = []

    for name in names:
        raw_str = name.strip()
        if not raw_str:
            cleaned_name_norm.append("")
            cleaned_name_ascii.append("")
            cleaned_legal_form.append("missing")
            continue

        clean_str = _APOSTROPHE_REGEX.sub('', raw_str.lower())
        clean_str = _PUNCT_REGEX.sub(' ', clean_str)
        clean_str = _WHITESPACE_REGEX.sub(' ', clean_str).strip()

        matches = _LEGAL_REGEX.findall(clean_str)
        if matches:
            seen = set()
            forms = []
            for m in matches:
                canon = LEGAL_FORMS_MAP[m.lower()]
                if canon not in seen:
                    seen.add(canon)
                    forms.append(canon)
            legal_str = ', '.join(forms)
        else:
            legal_str = "missing"

        name_clean = _LEGAL_REGEX.sub(' ', clean_str)
        norm = _WHITESPACE_REGEX.sub(' ', name_clean).strip() or clean_str

        # ASCII normalization
        try:
            import unidecode
            raw_ascii = unidecode.unidecode(norm).lower()
        except Exception:
            import unicodedata
            raw_ascii = unicodedata.normalize('NFKD', norm).encode('ascii', 'ignore').decode('utf-8').lower()

        ascii_clean = _WHITESPACE_REGEX.sub(' ', _ASCII_CLEAN_REGEX.sub(' ', raw_ascii)).strip() or norm

        cleaned_name_norm.append(norm)
        cleaned_name_ascii.append(ascii_clean)
        cleaned_legal_form.append(legal_str)

    for addr in addresses:
        cleaned_addr = _APOSTROPHE_REGEX.sub('', addr.lower())
        cleaned_addr = _PUNCT_REGEX.sub(' ', cleaned_addr)
        cleaned_address_norm.append(_WHITESPACE_REGEX.sub(' ', cleaned_addr).strip())

    for c in countries:
        cleaned_country_norm.append(c.strip().lower())

    processed_df = pd.DataFrame({
        "entity_id": df["entity_id"].values,
        "name_norm": cleaned_name_norm,
        "name_ascii": cleaned_name_ascii,
        "address_norm": cleaned_address_norm,
        "country_norm": cleaned_country_norm,
        "legal_form": cleaned_legal_form,
    }, index=df["entity_id"].values)

    return processed_df

print("[INFO] Preprocessing pipeline defined.")


In [ ]:
print("[INFO] 1. Loading and Preprocessing Test Source 1 (Queries)...")
t0 = time.time()
raw_test_s1 = pd.read_csv(TEST_S1_PATH, sep="\t", nrows=SAMPLE_ROWS, dtype=str)
test_s1 = preprocess_test_dataframe(raw_test_s1)
print(f"       Done in {time.time()-t0:.1f}s | Source 1 Entities: {len(test_s1):,}")

print("[INFO] 2. Loading and Preprocessing Test Source 2 (Candidates)...")
t0 = time.time()
raw_test_s2 = pd.read_csv(TEST_S2_PATH, sep="\t", nrows=SAMPLE_ROWS, dtype=str)
test_s2 = preprocess_test_dataframe(raw_test_s2)
print(f"       Done in {time.time()-t0:.1f}s | Source 2 Entities: {len(test_s2):,}")

print("[INFO] 3. Loading and Preprocessing Test Source 3 (Candidates)...")
t0 = time.time()
raw_test_s3 = pd.read_csv(TEST_S3_PATH, sep="\t", nrows=SAMPLE_ROWS, dtype=str)
test_s3 = preprocess_test_dataframe(raw_test_s3)
print(f"       Done in {time.time()-t0:.1f}s | Source 3 Entities: {len(test_s3):,}")

print("\n--- Preprocessed Sample (Source 1) ---")
display(test_s1.head(3))


In [ ]:
print(f"[INFO] Loading model weights from: {MODEL_DIR.resolve()}")

tokenizer = CharTokenizer.load(MODEL_DIR / "tokenizer.json")
legal_vocab = LegalFormVocab.load(MODEL_DIR / "legal_form_vocab.json")

model = EntityEncoder(
    char_vocab_size=len(tokenizer.vocab) + 2,
    legal_vocab_size=len(legal_vocab.vocab) + 2
).to(device)

checkpoint = torch.load(MODEL_DIR / "best.pt", map_location=device)
model.load_state_dict(checkpoint["model"])
model.eval()

print(f"[INFO] Successfully restored model weights from Epoch: {checkpoint.get('epoch', 'N/A')}")
print(f"[INFO] Model Parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
print("[INFO] Combining Source 2 and Source 3 candidates...")
candidate_catalog = pd.concat([test_s2, test_s3], ignore_index=True).drop_duplicates("entity_id").set_index("entity_id", drop=False)
candidate_ids = candidate_catalog["entity_id"].to_numpy()

print(f"[INFO] Computing embeddings for {len(candidate_catalog):,} candidates...")
t0 = time.time()
candidate_embeddings = generate_embeddings(
    model, tokenizer, legal_vocab, candidate_catalog, device=device, batch_size=512
)
print(f"[INFO] Candidate embeddings ready in {time.time()-t0:.1f}s | Matrix shape: {candidate_embeddings.shape}")


In [ ]:
def recommend_test_entity(query_id: str, top_k: int = 5):
    """Retrieves top matching candidate businesses for any given test Source 1 entity ID."""
    if query_id not in test_s1.index:
        print(f"[ERROR] Entity ID '{query_id}' not found in Source 1 test records.")
        return None

    query_record = test_s1.loc[[query_id]]
    query_vec = generate_embeddings(model, tokenizer, legal_vocab, query_record, device=device)

    # Cosine Similarity via Dot Product
    scores = (query_vec @ candidate_embeddings.T).flatten()
    top_indices = np.argsort(-scores)[:top_k]

    q = query_record.iloc[0]
    print("\n" + "=" * 70)
    print(f"QUERY BUSINESS ID : {query_id}")
    print(f"Name              : {q['name_norm']}")
    print(f"Address           : {q['address_norm']}")
    print(f"Country           : {q['country_norm']}")
    print(f"Legal Form        : {q['legal_form']}")
    print("=" * 70)
    print(f"TOP {top_k} RECOMMENDED MATCHES:")
    print("=" * 70)

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        cand_id = candidate_ids[idx]
        cand = candidate_catalog.iloc[idx]
        score = scores[idx]

        print(f"Rank {rank} | Score: {score:.4f}")
        print(f"  ID        : {cand_id}")
        print(f"  Name      : {cand['name_norm']}")
        print(f"  Address   : {cand['address_norm']}")
        print(f"  Country   : {cand['country_norm']}")
        print(f"  Legal Form: {cand['legal_form']}")
        print("-" * 70)

        results.append({
            "rank": rank,
            "candidate_id": cand_id,
            "similarity": round(float(score), 4),
            "name": cand["name_norm"],
            "address": cand["address_norm"],
            "country": cand["country_norm"]
        })

    return pd.DataFrame(results)

# Test with first Source 1 entity
sample_id = test_s1["entity_id"].iloc[0]
recommend_test_entity(sample_id, top_k=5)


In [ ]:
def generate_matching_results(query_df: pd.DataFrame, threshold: float = 0.65, top_k: int = 5) -> pd.DataFrame:
    """Generates matching_results.tsv (Source 1 ID -> comma-separated matched IDs)."""
    print(f"[INFO] Computing embeddings for {len(query_df):,} query records...")
    t0 = time.time()
    query_embeddings = generate_embeddings(
        model, tokenizer, legal_vocab, query_df, device=device, batch_size=512
    )
    print(f"[INFO] Query embeddings computed in {time.time()-t0:.1f}s")

    print("[INFO] Computing pairwise similarities and ranking...")
    all_scores = query_embeddings @ candidate_embeddings.T

    results = []
    q_ids = query_df["entity_id"].tolist()

    for i in range(len(q_ids)):
        scores_i = all_scores[i]
        top_idx = np.argsort(-scores_i)[:top_k]

        # Filter by minimum similarity threshold
        matched = [candidate_ids[idx] for idx in top_idx if scores_i[idx] >= threshold]

        results.append({
            "source1_entity_id": q_ids[i],
            "matched_entity_ids": ",".join(matched)
        })

    submission_df = pd.DataFrame(results)
    return submission_df

# Generate sample matching results
submission_sample = generate_matching_results(test_s1, threshold=0.65, top_k=5)

output_file = OUTPUT_DIR / "matching_results.tsv"
submission_sample.to_csv(output_file, sep="\t", index=False)
print(f"\n[INFO] Saved matching results to: {output_file.resolve()}")
print(f"[INFO] Total query rows: {len(submission_sample):,}")
print(f"[INFO] Queries with >=1 match: {(submission_sample['matched_entity_ids'] != '').sum():,}")

print("\n--- Preview of matching_results.tsv ---")
display(submission_sample.head(10))
